In [54]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/project_thinkfield/tinyrecursivemodels")
os.environ['DISABLE_COMPILE'] = '1'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
  Using cached setuptools_scm-9.2.1-py3-none-any.whl.metadata (7.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.7 MB/s eta 0:00:00
Using cached setuptools_scm-9.2.1-py3-none-any.whl (62 kB)
  Created wheel for adam-atan2: filename=adam_atan2-0.0.3-cp312-cp312-linux_x86_64.whl size=196278 sha256=6734be1ccc9230579dbb45ec1cc269acb0c12dc19471c4da24aff76e3d43a29f
  Stored in directory: /root/.cache/pip/wheels/45/32/31/044d4e2d7dc14d2820c0232d9f41b3ae8cb3d8451451a80dbf
Successfully built adam-atan2


In [33]:
import os
import json
from glob import glob
import hashlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry

import torch
import torch.nn.functional as F
import numpy as np
from numba import njit
import yaml
from omegaconf import OmegaConf
from tqdm import tqdm
import math
import types

from puzzle_dataset import PuzzleDatasetMetadata # puzzle_dataset에서 가져올 것은 PuzzleDatasetMetadata입니다.
from pretrain import PretrainConfig, create_model, create_dataloader # create_dataloader는 pretrain에서 가져옵니다.
from dataset.common import inverse_dihedral_transform


#DATASET_PATH = "data/arc-aug-1000"  # ARC-1
#DATASET_PATH = "data/arc-2-aug-1000"  # ARC-2
DATASET_PATH = "data/arc_mini_arc2"  # ARC-2 mini


CHECKPOINT_PATH = "checkpoints/Arc_mini_arc2-ACT-torch/TinyRecursiveReasoningModel_ACTV1 thistle-giraffe/step_362160"
PAD_PUZZLE_IDENTIFIER = 0

# Visualization
ARC_COLOR_MAP = mcolors.ListedColormap([
    "#000000",  # symbol_0: black
    "#0074D9",  # symbol_1: blue
    "#FF4136",  # symbol_2: red
    "#2ECC40",  # symbol_3: green
    "#FFDC00",  # symbol_4: yellow
    "#AAAAAA",  # symbol_5: grey
    "#F012BE",  # symbol_6: fuschia
    "#FF851B",  # symbol_7: orange
    "#7FDBFF",  # symbol_8: teal
    "#870C25"   # symbol_9: brown
])

In [55]:
def inverse_aug(name: str, grid: np.ndarray):
    if "_" not in name: return grid
    trans_id, perm = name.split("_")[-2:]
    trans_id = int(trans_id[1:])
    inv_perm = np.argsort(list(perm))
    return inv_perm[inverse_dihedral_transform(grid, trans_id)]

def grid_hash(grid: np.ndarray):
    return hash((grid.tobytes(), grid.shape))

@njit
def crop(grid: np.ndarray):
    if grid.ndim == 1: grid = grid.reshape(30, 30)
    max_area = 0
    max_size = (0, 0)
    nr, nc = grid.shape
    num_c = nc
    for num_r in range(1, nr + 1):
        for c in range(1, num_c + 1):
            x = grid[num_r - 1, c - 1]
            if x <= 1:
                num_c = c - 1
                break
        area = num_r * num_c
        if area > max_area:
            max_area = area
            max_size = (num_r, num_c)
    return grid[:max_size[0], :max_size[1]]

def test_and_visualize(Ks=[1, 3]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    with open(os.path.join(os.path.dirname(CHECKPOINT_PATH), "all_config.yaml"), "r") as f:
        config = PretrainConfig(**yaml.safe_load(f))

    config.data_paths = [DATASET_PATH]
    config.data_paths_test = [DATASET_PATH]
    save_outputs = ["inputs", "labels", "puzzle_identifiers", "logits", "q_halt_logits"]
    print("Starting evaluation")

    _, eval_metadata = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
    eval_loader, _ = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
    model, _, _ = create_model(config, eval_metadata, rank=0, world_size=1)

    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    if any(key.startswith("_orig_mod.") for key in model.state_dict().keys()) and not any(key.startswith("_orig_mod.") for key in state_dict.keys()):
        state_dict = {"_orig_mod." + k: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)

    model.to(device)
    model.eval()

    print(model)

    all_outputs = {key: [] for key in save_outputs}
    eval_loader.dataset._lazy_load_dataset()
    total_examples = sum(len(d["inputs"]) for d in eval_loader.dataset._data.values())
    total_batches = math.ceil(total_examples / config.global_batch_size)
    with torch.inference_mode():
        with tqdm(total=total_batches, desc="Evaluating") as pbar:
            for set_name, batch, global_batch_size in eval_loader:
                if global_batch_size == 0: continue

                pbar.set_description(f"Evaluating (Set: {set_name})")

                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

                carry = model.initial_carry(batch)

                while True:
                    carry, loss, metrics, preds, all_finish = model(carry=carry, batch=batch, return_keys=set(save_outputs))
                    if all_finish:
                        break
                outputs = {**batch, **preds}
                for key in save_outputs:
                    if key in outputs:
                        all_outputs[key].append(outputs[key].cpu())

                pbar.update(1)

    all_preds = {key: torch.cat(all_outputs[key], dim=0) for key in save_outputs if all_outputs[key]}

    with open(os.path.join(DATASET_PATH, "identifiers.json"), "r") as f:
        identifier_map = {int(k): v for k, v in json.load(f).items()}

    mask = all_preds["puzzle_identifiers"] != PAD_PUZZLE_IDENTIFIER
    all_preds = {k: v[mask] for k, v in all_preds.items()}
    global_hmap = {}
    puzzle_labels = {}
    for identifier, input_tensor, label_tensor in zip(all_preds["puzzle_identifiers"], all_preds["inputs"], all_preds["labels"]):
        name = identifier_map[identifier.item()]
        if "_" not in name:
            puzzle_labels.setdefault(name, {})
            input_np = crop(input_tensor.numpy())
            label_np = crop(label_tensor.numpy())
            input_hash = grid_hash(input_np)
            label_hash = grid_hash(label_np)
            global_hmap[input_hash] = input_np
            global_hmap[label_hash] = label_np
            if input_hash not in puzzle_labels[name]:
                puzzle_labels[name][input_hash] = label_hash
    print(f"Number of original puzzles found: {len(puzzle_labels)}")
    preds = all_preds["logits"].argmax(-1)
    pred_answers = {}
    for identifier, input_tensor, pred_tensor, q_tensor in zip(all_preds["puzzle_identifiers"], all_preds["inputs"], preds, all_preds["q_halt_logits"].sigmoid()):
        name = identifier_map[identifier.item()]
        orig_name = name.split("_")[0]
        if orig_name not in puzzle_labels: continue
        input_np = crop(input_tensor.numpy())
        input_hash = grid_hash(inverse_aug(name, input_np))
        if input_hash not in puzzle_labels[orig_name]: continue
        pred_np = inverse_aug(name, crop(pred_tensor.numpy()))
        pred_hash = grid_hash(pred_np)
        global_hmap[pred_hash] = pred_np
        pred_answers.setdefault(orig_name, {})
        pred_answers[orig_name].setdefault(input_hash, [])
        pred_answers[orig_name][input_hash].append((pred_hash, q_tensor.item()))

    correct = [0] * len(Ks)
    for name, tests in puzzle_labels.items():
        num_test_correct = [0] * len(Ks)
        for input_hash, label_hash in tests.items():
            if name in pred_answers and input_hash in pred_answers[name]:
                p = pred_answers[name][input_hash]
                p_map = {}
                for h, q in p:
                    p_map.setdefault(h, [0, 0.0])
                    p_map[h][0] += 1
                    p_map[h][1] += q
                p_map = {h: (stats[0], stats[1]/stats[0]) for h, stats in p_map.items()}
                p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1][1], reverse=True)
                for i, k in enumerate(Ks):
                    ok = any(h == label_hash for h, stats in p_map_sorted[:k])
                    num_test_correct[i] += ok
        for i in range(len(Ks)):
            if len(tests) > 0 and num_test_correct[i] == len(tests):
                correct[i] += 1

    if len(puzzle_labels) > 0:
        for i, k in enumerate(Ks):
            print(f"Top-{k} Accuracy: {correct[i] / len(puzzle_labels) * 100:.2f}%")

    for puzzle_name, tests in tqdm(puzzle_labels.items(), desc="Visualizing puzzles"):
        num_tests = len(tests)
        fig, axes = plt.subplots(num_tests, 2 + Ks[-1], figsize=(4 * (2+Ks[-1]), num_tests * 4), squeeze=False)
        fig.suptitle(f"Puzzle: {puzzle_name}", fontsize=16)
        p_map_sorted_for_viz = []
        for i, (input_hash, label_hash) in enumerate(tests.items()):
            axes[i, 0].imshow(global_hmap[input_hash], cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
            axes[i, 0].set_title("Input")
            axes[i, 0].axis('off')
            axes[i, 1].imshow(global_hmap[label_hash], cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
            axes[i, 1].set_title("Correct Answer")
            axes[i, 1].axis('off')
            if puzzle_name in pred_answers and input_hash in pred_answers[puzzle_name]:
                p = pred_answers[puzzle_name][input_hash]
                p_map = {}
                for h, q in p:
                    p_map.setdefault(h, [0, 0.0])
                    p_map[h][0] += 1
                    p_map[h][1] += q
                p_map = {h: (stats[0], stats[1]/stats[0]) for h, stats in p_map.items()}
                p_map_sorted_for_viz = sorted(p_map.items(), key=lambda kv: kv[1][1], reverse=True)
                for j, (h, stats) in enumerate(p_map_sorted_for_viz[:Ks[-1]]):
                    pred_grid = global_hmap[h]
                    axes[i, 2+j].imshow(pred_grid, cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
                    title = f"Pred #{j+1}\\n(Score: {stats[1]:.2f}, Votes: {stats[0]})"
                    if h == label_hash:
                        axes[i, 2+j].spines[:].set_color('green')
                        axes[i, 2+j].spines[:].set_linewidth(4)
                    axes[i, 2+j].set_title(title)
                    axes[i, 2+j].axis('off')

            for j in range(len(p_map_sorted_for_viz), Ks[-1]):
                axes[i, 2+j].axis('off')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

# --- 코드 실행 ---
test_and_visualize()

Using device: cuda
Starting evaluation
TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)
ACTLossHead(
  (model): TinyRecursiveReasoningModel_ACTV1(
    (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
      (embed_tokens): CastedEmbedding()
      (lm_head): CastedLinear()
      (q_head): CastedLinear()
      (puzzle_emb): CastedSparseEmbedding()
      (rotary_e

Evaluating (Set: all):   0%|          | 0/131 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [51]:
def inverse_aug(name: str, grid: np.ndarray):
    if "_" not in name: return grid
    trans_id, perm = name.split("_")[-2:]
    trans_id = int(trans_id[1:])
    inv_perm = np.argsort(list(perm))
    return inv_perm[inverse_dihedral_transform(grid, trans_id)]

def grid_hash(grid: np.ndarray):
    return hash((grid.tobytes(), grid.shape))

@njit
def crop(grid: np.ndarray):
    if grid.ndim == 1: grid = grid.reshape(30, 30)
    max_area = 0
    max_size = (0, 0)
    nr, nc = grid.shape
    num_c = nc
    for num_r in range(1, nr + 1):
        for c in range(1, num_c + 1):
            x = grid[num_r - 1, c - 1]
            if x <= 1:
                num_c = c - 1
                break
        area = num_r * num_c
        if area > max_area:
            max_area = area
            max_size = (num_r, num_c)
    return grid[:max_size[0], :max_size[1]]

# --- (핵심 수정) ---
# 1. 버그가 수정된 새로운 reset_carry 함수를 정의합니다.
def patched_reset_carry(self, reset_flag: torch.Tensor, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry):
    device = carry.z_H.device
    reset_flag_on_device = reset_flag.to(device)
    h_init = self.H_init.to(device)
    l_init = self.L_init.to(device)
    return TinyRecursiveReasoningModel_ACTV1InnerCarry(
        z_H=torch.where(reset_flag_on_device.view(-1, 1, 1), h_init, carry.z_H),
        z_L=torch.where(reset_flag_on_device.view(-1, 1, 1), l_init, carry.z_L),
    )

def patched_forward(self, carry: TinyRecursiveReasoningModel_ACTV1Carry, batch: dict):
    # (버그 수정) carry.halted 텐서를 GPU로 이동
    device = next(self.parameters()).device
    halted_on_device = carry.halted.to(device)

    new_steps = torch.where(halted_on_device, 0, carry.steps)
    new_current_data = {k: torch.where(halted_on_device.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

    # 원본 forward 함수의 나머지 로직
    new_inner_carry, outputs = self.inner(carry.inner_carry, new_current_data)
    new_halted = halted_on_device | outputs.halt

    return TinyRecursiveReasoningModel_ACTV1Carry(
        steps=new_steps + 1,
        halted=new_halted,
        inner_carry=new_inner_carry,
        current_data=new_current_data,
    ), outputs
# --- (핵심 수정 끝) ---

def test_and_visualize(Ks=[1, 3]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    with open(os.path.join(os.path.dirname(CHECKPOINT_PATH), "all_config.yaml"), "r") as f:
        config = PretrainConfig(**yaml.safe_load(f))

    config.data_paths = [DATASET_PATH]
    config.data_paths_test = [DATASET_PATH]
    save_outputs = ["inputs", "labels", "puzzle_identifiers", "logits", "q_halt_logits"]
    print("Starting evaluation")

    _, eval_metadata = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
    eval_loader, _ = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
    model, _, _ = create_model(config, eval_metadata, rank=0, world_size=1)

    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    if any(key.startswith("_orig_mod.") for key in model.state_dict().keys()) and not any(key.startswith("_orig_mod.") for key in state_dict.keys()):
        state_dict = {"_orig_mod." + k: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)

    # --- (핵심 수정) ---
    # 2. 모델의 문제가 되는 함수를 우리가 만든 수정된 함수로 교체합니다 (몽키 패치).
    inner_model = model.model.inner
    inner_model.reset_carry = types.MethodType(patched_reset_carry, inner_model)
    print("Model `reset_carry` function has been patched in memory.")
    # --- (핵심 수정 끝) ---

    model.to(device)
    model.eval()

    # (이후 코드는 이전과 동일)
    all_outputs = {key: [] for key in save_outputs}
    eval_loader.dataset._lazy_load_dataset()
    total_examples = sum(len(d["inputs"]) for d in eval_loader.dataset._data.values())
    total_batches = math.ceil(total_examples / config.global_batch_size)
    with torch.inference_mode():
        # --- (tqdm 시각화 개선) ---
        # 2. 미리 계산한 total_batches를 tqdm에 전달하고, with 구문을 사용해 깔끔하게 관리합니다.
        with tqdm(total=total_batches, desc="Evaluating") as pbar:
            for set_name, batch, global_batch_size in eval_loader:
                if global_batch_size == 0: continue

                # 3. 현재 처리 중인 set_name을 진행률 표시줄에 업데이트합니다.
                pbar.set_description(f"Evaluating (Set: {set_name})")

                batch = {k: v.to(device) for k, v in batch.items()}
                carry = model.initial_carry(batch)
                while True:
                    carry, loss, metrics, preds, all_finish = model(carry=carry, batch=batch, return_keys=set(save_outputs))
                    if all_finish:
                        break
                outputs = {**batch, **preds}
                for key in save_outputs:
                    if key in outputs:
                        all_outputs[key].append(outputs[key].cpu())

                # 4. 배치 하나 처리가 끝날 때마다 진행률을 1 올립니다.
                pbar.update(1)
    all_preds = {key: torch.cat(all_outputs[key], dim=0) for key in save_outputs if all_outputs[key]}

    with open(os.path.join(DATASET_PATH, "identifiers.json"), "r") as f:
        identifier_map = {int(k): v for k, v in json.load(f).items()}

    mask = all_preds["puzzle_identifiers"] != PAD_PUZZLE_IDENTIFIER
    all_preds = {k: v[mask] for k, v in all_preds.items()}
    global_hmap = {}
    puzzle_labels = {}
    for identifier, input_tensor, label_tensor in zip(all_preds["puzzle_identifiers"], all_preds["inputs"], all_preds["labels"]):
        name = identifier_map[identifier.item()]
        if "_" not in name:
            puzzle_labels.setdefault(name, {})
            input_np = crop(input_tensor.numpy())
            label_np = crop(label_tensor.numpy())
            input_hash = grid_hash(input_np)
            label_hash = grid_hash(label_np)
            global_hmap[input_hash] = input_np
            global_hmap[label_hash] = label_np
            if input_hash not in puzzle_labels[name]:
                puzzle_labels[name][input_hash] = label_hash
    print(f"Number of original puzzles found: {len(puzzle_labels)}")
    preds = all_preds["logits"].argmax(-1)
    pred_answers = {}
    for identifier, input_tensor, pred_tensor, q_tensor in zip(all_preds["puzzle_identifiers"], all_preds["inputs"], preds, all_preds["q_halt_logits"].sigmoid()):
        name = identifier_map[identifier.item()]
        orig_name = name.split("_")[0]
        if orig_name not in puzzle_labels: continue
        input_np = crop(input_tensor.numpy())
        input_hash = grid_hash(inverse_aug(name, input_np))
        if input_hash not in puzzle_labels[orig_name]: continue
        pred_np = inverse_aug(name, crop(pred_tensor.numpy()))
        pred_hash = grid_hash(pred_np)
        global_hmap[pred_hash] = pred_np
        pred_answers.setdefault(orig_name, {})
        pred_answers[orig_name].setdefault(input_hash, [])
        pred_answers[orig_name][input_hash].append((pred_hash, q_tensor.item()))

    correct = [0] * len(Ks)
    for name, tests in puzzle_labels.items():
        num_test_correct = [0] * len(Ks)
        for input_hash, label_hash in tests.items():
            if name in pred_answers and input_hash in pred_answers[name]:
                p = pred_answers[name][input_hash]
                p_map = {}
                for h, q in p:
                    p_map.setdefault(h, [0, 0.0])
                    p_map[h][0] += 1
                    p_map[h][1] += q
                p_map = {h: (stats[0], stats[1]/stats[0]) for h, stats in p_map.items()}
                p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1][1], reverse=True)
                for i, k in enumerate(Ks):
                    ok = any(h == label_hash for h, stats in p_map_sorted[:k])
                    num_test_correct[i] += ok
        for i in range(len(Ks)):
            if len(tests) > 0 and num_test_correct[i] == len(tests):
                correct[i] += 1

    if len(puzzle_labels) > 0:
        for i, k in enumerate(Ks):
            print(f"Top-{k} Accuracy: {correct[i] / len(puzzle_labels) * 100:.2f}%")

    for puzzle_name, tests in tqdm(puzzle_labels.items(), desc="Visualizing puzzles"):
        num_tests = len(tests)
        fig, axes = plt.subplots(num_tests, 2 + Ks[-1], figsize=(4 * (2+Ks[-1]), num_tests * 4), squeeze=False)
        fig.suptitle(f"Puzzle: {puzzle_name}", fontsize=16)
        p_map_sorted_for_viz = []
        for i, (input_hash, label_hash) in enumerate(tests.items()):
            axes[i, 0].imshow(global_hmap[input_hash], cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
            axes[i, 0].set_title("Input")
            axes[i, 0].axis('off')
            axes[i, 1].imshow(global_hmap[label_hash], cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
            axes[i, 1].set_title("Correct Answer")
            axes[i, 1].axis('off')
            if puzzle_name in pred_answers and input_hash in pred_answers[puzzle_name]:
                p = pred_answers[puzzle_name][input_hash]
                p_map = {}
                for h, q in p:
                    p_map.setdefault(h, [0, 0.0])
                    p_map[h][0] += 1
                    p_map[h][1] += q
                p_map = {h: (stats[0], stats[1]/stats[0]) for h, stats in p_map.items()}
                p_map_sorted_for_viz = sorted(p_map.items(), key=lambda kv: kv[1][1], reverse=True)
                for j, (h, stats) in enumerate(p_map_sorted_for_viz[:Ks[-1]]):
                    pred_grid = global_hmap[h]
                    axes[i, 2+j].imshow(pred_grid, cmap=ARC_COLOR_MAP, vmin=0, vmax=9)
                    title = f"Pred #{j+1}\n(Score: {stats[1]:.2f}, Votes: {stats[0]})"
                    if h == label_hash:
                        axes[i, 2+j].spines[:].set_color('green')
                        axes[i, 2+j].spines[:].set_linewidth(4)
                    axes[i, 2+j].set_title(title)
                    axes[i, 2+j].axis('off')

            for j in range(len(p_map_sorted_for_viz), Ks[-1]):
                axes[i, 2+j].axis('off')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

# --- 코드 실행 ---
test_and_visualize()

Using device: cuda
Starting evaluation
TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)
Model `reset_carry` function has been patched in memory.


Evaluating (Set: all):   0%|          | 0/131 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [21]:
!git add .

^C
